In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'
OUTPUT_DIR = 'results'

In [4]:
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok = True)

In [5]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [6]:
metric_cols = [
    "Accuracy",
    "Weighted Accuracy",
    "Time",
    "Monotonicity",
    "Separability",
    "Linearity"
]

In [7]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    if 'dementia' not in file:

        # Process texts.
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w.lower() not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=['Total Documents', 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,0.498065,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,0.398338,4.360843,0.035360,0.267681,46.278934,17.0
4,huffPostNews,0.390427,4.199670,0.023536,0.238591,25.163032,23.0
5,medicalAbstracts,0.308262,5.109807,0.021201,0.263213,205.660064,200.0
6,simSUM,0.124588,4.106722,0.012611,0.381137,104.821400,103.0
7,syntheticCareHomeNurseNotes,0.340652,4.937623,0.027334,0.277734,26.532596,24.0
8,yahoo,0.424969,3.922505,0.029172,0.255604,47.845493,42.0


In [8]:
def make_non_normalized_dfs(input_folder, output_file_name):
    all_temp_dfs = []
    for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}'):
        if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}'):
            combination_splits = combination.split('_')
            dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}/{combination}_ksc_metrics_measures.csv')
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Dataset 1'] = dataset1
            grouped_df['Dataset 2'] = dataset2
            grouped_df['Repetitions'] = repetitions
            grouped_df['Metric'] = grouped_df.index
            grouped_df.reset_index(inplace=True)
            grouped_df.drop(columns='metric', inplace=True)
            all_temp_dfs.append(grouped_df)

    all_dfs = pd.concat(all_temp_dfs)

    print(# All datasets should have been compared the same number of times for this section to work.
    Counter(list(all_dfs['Dataset 1']) + list(all_dfs['Dataset 2'])))

    temp_dataset_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = all_dfs[
                (all_dfs['Dataset 1'] == dataset) | 
                (all_dfs['Dataset 2'] == dataset)
            ].copy()
        temp_df = temp_df.groupby('Metric')[metric_cols].mean()
        temp_df['Dataset'] = dataset
        temp_dataset_dfs.append(temp_df)
    dataset_df = pd.concat(temp_dataset_dfs)
    dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}.xlsx')

    mean_dataset_df = dataset_df.groupby('Metric')[metric_cols].mean()
    mean_dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}Mean.xlsx')

    return dataset_df, mean_dataset_df

In [9]:
ksc_dataset_df, ksc_mean_dataset_df = make_non_normalized_dfs('ksc', 'kscDataset')
ksc_synth_dataset_df, ksc_synth_mean_dataset_df = make_non_normalized_dfs('ksc_synth', 'kscSynthDataset')

Counter({'atis': 160, 'banking77': 160, 'clinc150': 160, 'clinicalDialogueSummarizations': 160, 'huffPostNews': 160, 'medicalAbstracts': 160, 'simSUM': 160, 'syntheticCareHomeNurseNotes': 160, 'yahoo': 160})
Counter({'atis': 40, 'banking77': 40, 'clinc150': 40, 'clinicalDialogueSummarizations': 40, 'huffPostNews': 40, 'medicalAbstracts': 40, 'simSUM': 40, 'syntheticCareHomeNurseNotes': 40, 'yahoo': 40})


In [10]:
ksc_dataset_df['Type'] = 'KSC'
ksc_synth_dataset_df['Type'] = 'KSC_Synth'

In [19]:
all_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Dataset', 'Metric'])[metric_cols].mean()
all_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Metric'])[metric_cols].mean()
all_type_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Metric'])[metric_cols].mean()
all_type_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Dataset', 'Metric'])[metric_cols].mean()

In [12]:
all_dataset_df.to_excel(f'./{OUTPUT_DIR}/allDataset.xlsx')
all_mean_df.to_excel(f'./{OUTPUT_DIR}/allDatasetMean.xlsx')

In [20]:
all_type_mean_df

Accuracy  Weighted Accuracy       Time  Monotonicity  \
Type      Metric                                                              
KSC       CHI          0.003013           0.004436   2.049684     -0.723959   
          CLASSIFIER   0.857003           0.800860   3.271376      0.832664   
          DC           0.839487           0.784377   0.287120      0.816886   
          FID          0.945962           0.919281   0.052061      0.850500   
          IRPR         0.735493           0.688856  24.199156      0.528044   
          MAUVE        0.960912           0.938161   0.034817      0.939551   
          PR           0.676019           0.630614   0.287167      0.505978   
          TRADITIONAL  0.923040           0.890403   0.007801      0.893365   
          ZERO         0.947468           0.921543   0.001043      0.928363   
          ZIPF         0.747523           0.716545   3.600523      0.470527   
KSC_Synth CHI          0.000872           0.001404   2.407829     -0.682752   
          CLASSIFIER   0.790482           0.726319   3.232378      0.753348   
          DC           0.856540           0.806411   0.287427      0.831572   
          FID          0.861703           0.810074   0.053042      0.777644   
          IRPR         0.736819           0.682549  24.434501      0.560124   
          MAUVE        0.833163           0.775197   0.035135      0.812641   
          PR           0.745286           0.687186   0.287481      0.613901   
          TRADITIONAL  0.866206           0.815964   0.008393      0.844089   
          ZERO         0.859017           0.811942   0.001171      0.813920   
          ZIPF         0.837577           0.799773   4.143529      0.713416   

                       Separability  Linearity  
Type      Metric                                
KSC       CHI              0.666252  -0.650407  
          CLASSIFIER       0.673873   0.843690  
          DC               0.821180   0.856057  
          FID              0.774550   0.887513  
          IRPR             0.333963   0.598514  
          MAUVE            0.894077   0.942962  
          PR               0.576051   0.683010  
          TRADITIONAL      0.821618   0.915873  
          ZERO             0.882241   0.945526  
          ZIPF             0.132102   0.460451  
KSC_Synth CHI              0.855849  -0.599414  
          CLASSIFIER       0.592896   0.787446  
          DC               0.810100   0.880150  
          FID              0.638276   0.817403  
          IRPR             0.307037   0.618082  
          MAUVE            0.761619   0.841320  
          PR               0.554908   0.708558  
          TRADITIONAL      0.737854   0.873278  
          ZERO             0.704574   0.846281  
          ZIPF             0.529558   0.740174

In [18]:
all_type_dataset_df

Accuracy  Weighted Accuracy       Time  \
Type      Dataset Metric                                                
KSC       atis    CHI          0.004628           0.007064   2.926928   
                  CLASSIFIER   0.867315           0.814681   3.294585   
                  DC           0.840625           0.783574   0.287123   
                  FID          0.969957           0.953420   0.052634   
                  IRPR         0.732247           0.685910  24.413256   
...                                 ...                ...        ...   
KSC_Synth yahoo   MAUVE        0.835300           0.769118   0.034947   
                  PR           0.765859           0.702131   0.287341   
                  TRADITIONAL  0.897227           0.849912   0.006917   
                  ZERO         0.864657           0.804562   0.001009   
                  ZIPF         0.918999           0.879252   2.932961   

                               Monotonicity  Separability  Linearity  
Type      Dataset Metric                                              
KSC       atis    CHI             -0.761667      0.555298  -0.693813  
                  CLASSIFIER       0.839550      0.676110   0.847141  
                  DC               0.808979      0.821679   0.855380  
                  FID              0.883610      0.834501   0.914207  
                  IRPR             0.538926      0.412263   0.611935  
...                                     ...           ...        ...  
KSC_Synth yahoo   MAUVE            0.832913      0.777003   0.843836  
                  PR               0.621249      0.520035   0.714203  
                  TRADITIONAL      0.860077      0.767610   0.890719  
                  ZERO             0.838057      0.719378   0.866630  
                  ZIPF             0.913932      0.848629   0.934140  

[180 rows x 6 columns]

In [13]:
all_dataset_df

Accuracy  Weighted Accuracy       Time  Monotonicity  \
Dataset Metric                                                              
atis    CHI          0.002760           0.004263   3.661326     -0.785576   
        CLASSIFIER   0.832297           0.769365   3.322660      0.815044   
        DC           0.837264           0.785220   0.287517      0.808655   
        FID          0.934256           0.904584   0.052949      0.896539   
        IRPR         0.757867           0.700720  24.165496      0.654909   
...                       ...                ...        ...           ...   
yahoo   MAUVE        0.887589           0.839629   0.034509      0.879875   
        PR           0.736502           0.677774   0.287362      0.567274   
        TRADITIONAL  0.904655           0.864681   0.007055      0.867244   
        ZERO         0.895355           0.848325   0.000991      0.871050   
        ZIPF         0.807670           0.772819   2.993768      0.646990   

                     Separability  Linearity  
Dataset Metric                                
atis    CHI              0.659010  -0.694897  
        CLASSIFIER       0.664653   0.837478  
        DC               0.810773   0.864069  
        FID              0.867836   0.924841  
        IRPR             0.550409   0.721282  
...                           ...        ...  
yahoo   MAUVE            0.830764   0.889467  
        PR               0.509720   0.675163  
        TRADITIONAL      0.778640   0.894625  
        ZERO             0.780649   0.895937  
        ZIPF             0.468334   0.659586  

[90 rows x 6 columns]

In [14]:
all_mean_df

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity
Metric,,,,,,
CHI,0.001942,0.002920,2.228757,-0.703356,0.761050,-0.624910
CLASSIFIER,0.823742,0.763589,3.251877,0.793006,0.633384,0.815568
DC,0.848013,0.795394,0.287273,0.824229,0.815640,0.868104
FID,0.903832,0.864677,0.052551,0.814072,0.706413,0.852458
IRPR,0.736156,0.685702,24.316829,0.544084,0.320500,0.608298
MAUVE,0.897037,0.856679,0.034976,0.876096,0.827848,0.892141
PR,0.710652,0.658900,0.287324,0.559940,0.565479,0.695784
TRADITIONAL,0.894623,0.853184,0.008097,0.868727,0.779736,0.894576
ZERO,0.903242,0.866743,0.001107,0.871141,0.793407,0.895904
